# 100m Grid Dataset Generation (Seoul)

## 목적
- 서울시 전역을 100m × 100m 격자로 분할
- 이후 LST, 유동인구, 취약계층 등 모든 데이터를
  **동일한 기준 grid(cell_id)** 에 매핑하기 위한
  **기본 공간 단위(dataset)** 생성

## 산출물
- 100m Grid Dataset

  - 본 프로젝트에서 사용되는 모든 공간 데이터는 **EPSG:3857 기반 pixelCoordinates 방식으로 생성된 100m grid(cell_id)** 에 매핑된다.

  - Grid 생성 방식
    - Projection: EPSG:3857 (meter-based)
    - Resolution: 100m × 100m
    - cell_id: 픽셀 좌표(x, y) 기반 전역 고유 ID
    - 경계 처리: 서울 행정경계 기준 clipping

  - 특징
    - geometry 없이도 cell_id 기반 위치 재현 가능
    - 모든 파생 데이터(LST, 유동인구, 취약계층 등)의 기준 grid


In [15]:
# 기본 라이브러리
import os
import ee
import geemap
from dotenv import load_dotenv

# .env 로드
load_dotenv()

# Earth Engine 인증
ee.Authenticate()

# 프로젝트 ID 불러오기
project_id = os.getenv("EARTHENGINE_PROJECT")

# Earth Engine 초기화
ee.Initialize(project=project_id)

In [3]:
# =========================
# Global Configuration
# =========================

# 분석 대상 지역
TARGET_COUNTRY = "Republic of Korea"
TARGET_CITY = "Seoul"

# Grid 설정
GRID_SIZE = 100  # meters
PROJECTION = "EPSG:3857"

In [4]:
# =========================
# Load Seoul Boundary
# =========================

# GAUL Level-1 행정구역
gaul1 = ee.FeatureCollection("FAO/GAUL/2015/level1")

# 대한민국 필터링
korea = gaul1.filter(
    ee.Filter.eq("ADM0_NAME", TARGET_COUNTRY)
)

# 서울 필터링
seoul = korea.filter(
    ee.Filter.eq("ADM1_NAME", TARGET_CITY)
)

# 서울 geometry 가져오기
seoul_geometry = seoul.geometry()

In [5]:
# 지도 생성
Map = geemap.Map()
Map.centerObject(seoul, 10)

# 스타일 지정
style = {
    "color": "red",
    "fillColor": "00000000",
    "width": 2
}

# 레이어 추가
Map.addLayer(seoul.style(**style), {}, "Seoul Boundary")

Map

Map(center=[37.5380311832361, 127.00584943229981], controls=(WidgetControl(options=['position', 'transparent_b…

In [6]:
# =========================
# Build 100m Grid
# =========================

grid_scale = GRID_SIZE
proj = ee.Projection(PROJECTION).atScale(grid_scale)

# pixel 좌표 이미지 생성
coords = ee.Image.pixelCoordinates(proj)

# 전역적으로 유일한 cell_id 생성
cell_id_img = (
    coords.select("x")
    .multiply(1e7)
    .add(coords.select("y"))
    .toInt64()
)

# raster → vector (100m grid polygon)
seoul_grid_100m = cell_id_img.reduceToVectors(
    geometry=seoul_geometry,
    scale=grid_scale,
    geometryType="polygon",
    eightConnected=False,
    labelProperty="cell_id",
    reducer=ee.Reducer.countEvery()
).filterBounds(seoul_geometry)

print("Grid count:", seoul_grid_100m.size().getInfo())

Grid count: 92398


In [7]:
# =========================
# Add centroid & bbox info
# =========================

def add_cell_info(feature):
    geom = feature.geometry()

    centroid = geom.centroid(1)
    bounds = geom.bounds(1)
    coords = ee.List(bounds.coordinates().get(0))
    ll = ee.List(coords.get(0))
    ur = ee.List(coords.get(2))

    return feature.set({
        "center_lon": centroid.coordinates().get(0),
        "center_lat": centroid.coordinates().get(1),
        "min_lon": ll.get(0),
        "min_lat": ll.get(1),
        "max_lon": ur.get(0),
        "max_lat": ur.get(1),
    })

seoul_grid_info = seoul_grid_100m.map(add_cell_info)

print(seoul_grid_info.first().toDictionary().getInfo())

{'cell_id': 1411415045163, 'center_lat': 37.552140132436634, 'center_lon': 126.78956667327222, 'count': 1, 'max_lat': 37.55249622355975, 'max_lon': 126.79001583119751, 'min_lat': 37.55178404007387, 'min_lon': 126.78911751591339}


In [8]:
# =========================
# cell_id validation
# =========================

total_count = seoul_grid_info.size()
unique_cell_id_count = seoul_grid_info.aggregate_count_distinct("cell_id")

print("Total grid count:", total_count.getInfo())
print("Unique cell_id count:", unique_cell_id_count.getInfo())

Total grid count: 92398
Unique cell_id count: 92398


In [9]:
# 시각화용 샘플 (500개 정도)
grid_sample = seoul_grid_100m.limit(500)

Map = geemap.Map()
Map.centerObject(seoul, 11)

Map.addLayer(
    grid_sample,
    {"color": "blue"},
    "100m Grid (sample)"
)

Map.addLayer(
    seoul.style(**style),
    {},
    "Seoul Boundary"
)

Map

Map(center=[37.53803118323625, 127.00584943229947], controls=(WidgetControl(options=['position', 'transparent_…

In [11]:
# =========================
# Export Grid (CSV, no geometry)
# =========================

output_dir = "../data_processed/processed/grid"
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "seoul_grid_100m.csv")

export_grid_csv = seoul_grid_info.select([
    "cell_id",
    "center_lon",
    "center_lat",
    "min_lon",
    "min_lat",
    "max_lon",
    "max_lat"
])

print(f"Exporting CSV to {csv_path}...")
geemap.ee_export_vector(
    export_grid_csv,
    filename=csv_path
)

# GEE export 시 system:index → cell_id 컬럼명/형식 통일 (다른 데이터셋과 merge 가능하도록)
import pandas as pd
df = pd.read_csv(csv_path)

# system:index만 있고 cell_id가 없으면 "+141141+45163" → "1411415045163" 형태로 변환
if "system:index" in df.columns:
    if "cell_id" not in df.columns:
        def parse_system_index(s):
            parts = str(s).strip("+").split("+")
            if len(parts) >= 2:
                return parts[0] + "50" + parts[1]  # x + "50" + y
            return s
        df["cell_id"] = df["system:index"].apply(parse_system_index)
    df = df.drop(columns=["system:index"])

# cell_id를 문자열로 통일 (예: 1414035045016 형태)
df["cell_id"] = df["cell_id"].astype("int64").astype(str)
# cell_id를 첫 번째 컬럼으로
cols = ["cell_id"] + [c for c in df.columns if c != "cell_id"]
df = df[cols]
df.to_csv(csv_path, index=False)
print("CSV export complete. (cell_id 컬럼명·형식 통일 적용)")

Exporting CSV to ../data_processed/processed/grid/seoul_grid_100m.csv...
Generating URL ...
Please wait ...
Data downloaded to /Users/roel/Project/KT_Aivle/빅프로젝트/data_AI/data_ai/data_processed/processed/grid/seoul_grid_100m.csv
CSV export complete.


In [12]:
# =========================
# Export Grid (GeoJSON, with geometry)
# =========================

output_dir = "../data_processed/processed/grid"
os.makedirs(output_dir, exist_ok=True)

geojson_path = os.path.join(output_dir, "seoul_grid_100m.geojson")

print(f"Exporting GeoJSON to {geojson_path}...")
geemap.ee_export_vector(
    seoul_grid_100m,
    filename=geojson_path
)
print("GeoJSON export complete.")

Exporting GeoJSON to ../data_processed/processed/grid/seoul_grid_100m.geojson...
Generating URL ...
Please wait ...
Data downloaded to /Users/roel/Project/KT_Aivle/빅프로젝트/data_AI/data_ai/data_processed/processed/grid/seoul_grid_100m.geojson
GeoJSON export complete.


In [13]:
# =========================
# Export Grid as SHP (for GEE Asset upload)
# =========================

output_dir = "../data_processed/processed/grid"
os.makedirs(output_dir, exist_ok=True)

shp_path = os.path.join(output_dir, "seoul_grid_100m.shp")

print(f"Exporting SHP to {shp_path}...")
geemap.ee_export_vector(
    seoul_grid_100m,   # geometry 포함 grid
    filename=shp_path
)
print("SHP export complete.")

Exporting SHP to ../data_processed/processed/grid/seoul_grid_100m.shp...
Generating URL ...
Please wait ...
Data downloaded to /Users/roel/Project/KT_Aivle/빅프로젝트/data_AI/data_ai/data_processed/processed/grid/seoul_grid_100m.shp
SHP export complete.


In [14]:
import ee
print(ee.batch.Task.list())

[<Task RHXTSPMMCBVPESD2U63D6QFE EXPORT_FEATURES: seoul_grid_100m_shp (COMPLETED)>, <Task 2MF7HWIKUHATP6DEOVIOJQTX EXPORT_FEATURES: seoul_grid_100m_geojson (COMPLETED)>, <Task ZDA3QEMRRSCVPM4GSMMX7VSH EXPORT_FEATURES: seoul_grid_100m_cell_id (COMPLETED)>]
